# Smart Loan Recovery System with Machine Learning

This notebook provides an interactive analysis of the Smart Loan Recovery System.

## Overview
This system uses machine learning to:
- Segment borrowers based on risk profiles
- Predict high-risk borrowers
- Assign recovery strategies
- Optimize collection efforts

## 1. Import Libraries and Setup

In [ ]:
# Import required libraries
import sys
import os

# Add src directory to path
sys.path.append('../src')

from loan_recovery_system import SmartLoanRecoverySystem
from generate_sample_data import generate_loan_data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 2. Generate or Load Data

In [ ]:
# Generate sample data
print("Generating sample loan recovery data...")
df = generate_loan_data(1000)

# Save the data
data_path = "../data/loan_recovery_data.csv"
df.to_csv(data_path, index=False)

print(f"Data generated and saved to {data_path}")
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Initialize the Recovery System

In [ ]:
# Initialize the Smart Loan Recovery System
recovery_system = SmartLoanRecoverySystem(data_path)

print("Smart Loan Recovery System initialized successfully!")

## 4. Exploratory Data Analysis

In [ ]:
# Perform exploratory data analysis
recovery_system.explore_data()

## 5. Borrower Segmentation using K-Means Clustering

In [ ]:
# Perform borrower segmentation
recovery_system.perform_clustering(n_clusters=4)

## 6. Risk Prediction Model

In [ ]:
# Build risk prediction model
model = recovery_system.build_risk_prediction_model()

print("\nModel training completed!")

## 7. Recovery Strategy Assignment

In [ ]:
# Assign recovery strategies
sample_strategies = recovery_system.assign_recovery_strategies()

print("\nSample Recovery Strategies:")
sample_strategies

## 8. Key Insights and Analysis

In [ ]:
# Generate insights
recovery_system.generate_insights()

## 9. Custom Analysis - Risk Score Distribution by Segment

In [ ]:
# Custom visualization - Risk scores by segment
df_with_results = recovery_system.df

if 'Risk_Score' in df_with_results.columns:
    plt.figure(figsize=(12, 8))
    
    # Box plot
    plt.subplot(2, 2, 1)
    sns.boxplot(data=df_with_results, x='Segment_Name', y='Risk_Score')
    plt.title('Risk Score Distribution by Segment')
    plt.xticks(rotation=45)
    
    # Violin plot
    plt.subplot(2, 2, 2)
    sns.violinplot(data=df_with_results, x='Segment_Name', y='Risk_Score')
    plt.title('Risk Score Density by Segment')
    plt.xticks(rotation=45)
    
    # Strategy distribution
    plt.subplot(2, 2, 3)
    strategy_counts = df_with_results['Recovery_Strategy'].value_counts()
    plt.pie(strategy_counts.values, labels=strategy_counts.index, autopct='%1.1f%%')
    plt.title('Recovery Strategy Distribution')
    
    # Risk score histogram
    plt.subplot(2, 2, 4)
    plt.hist(df_with_results['Risk_Score'], bins=30, alpha=0.7, edgecolor='black')
    plt.xlabel('Risk Score')
    plt.ylabel('Frequency')
    plt.title('Overall Risk Score Distribution')
    
    plt.tight_layout()
    plt.show()
else:
    print("Risk scores not available. Please run the model first.")

## 10. Interactive Dashboard

In [ ]:
# Create interactive dashboard
if 'Risk_Score' in df_with_results.columns:
    # Interactive scatter plot
    fig = px.scatter(df_with_results, 
                    x='Monthly_Income', 
                    y='Outstanding_Loan_Amount',
                    size='Risk_Score',
                    color='Recovery_Strategy',
                    hover_data=['Age', 'Loan_Amount', 'Segment_Name'],
                    title='Interactive Loan Recovery Dashboard')
    
    fig.update_layout(height=600)
    fig.show()
    
    # Risk score vs Recovery outcome
    fig2 = px.box(df_with_results, 
                  x='Recovery_Status', 
                  y='Risk_Score',
                  title='Risk Score by Recovery Status')
    fig2.show()
else:
    print("Risk scores not available. Please run the model first.")

## 11. Model Performance Analysis

In [ ]:
# Detailed model performance analysis
if recovery_system.rf_model is not None:
    # Feature importance
    feature_importance = pd.DataFrame({
        'Feature': recovery_system.feature_names,
        'Importance': recovery_system.rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance Ranking:")
    print(feature_importance)
    
    # Plot feature importance
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance, x='Importance', y='Feature')
    plt.title('Feature Importance for Risk Prediction')
    plt.tight_layout()
    plt.show()
else:
    print("Model not trained yet. Please run the risk prediction model first.")

## 12. Save Results

In [ ]:
# Save all results
recovery_system.save_results()

print("\n" + "="*60)
print("ANALYSIS COMPLETED SUCCESSFULLY!")
print("Check the outputs/ directory for detailed results.")
print("="*60)

## 13. Summary Statistics

In [ ]:
# Final summary
if 'Risk_Score' in recovery_system.df.columns:
    summary_stats = {
        'Total Borrowers': len(recovery_system.df),
        'High Risk Borrowers (>0.75)': len(recovery_system.df[recovery_system.df['Risk_Score'] > 0.75]),
        'Medium Risk Borrowers (0.5-0.75)': len(recovery_system.df[
            (recovery_system.df['Risk_Score'] >= 0.5) & (recovery_system.df['Risk_Score'] <= 0.75)
        ]),
        'Low Risk Borrowers (<0.5)': len(recovery_system.df[recovery_system.df['Risk_Score'] < 0.5]),
        'Average Risk Score': recovery_system.df['Risk_Score'].mean().round(3),
        'Total Outstanding Amount': f"${recovery_system.df['Outstanding_Loan_Amount'].sum():,.2f}",
        'Average Outstanding per Borrower': f"${recovery_system.df['Outstanding_Loan_Amount'].mean():,.2f}"
    }
    
    print("\n=== FINAL SUMMARY STATISTICS ===")
    for key, value in summary_stats.items():
        print(f"{key}: {value}")
        
    # Recovery strategies summary
    print("\n=== RECOVERY STRATEGIES SUMMARY ===")
    strategy_summary = recovery_system.df['Recovery_Strategy'].value_counts()
    for strategy, count in strategy_summary.items():
        percentage = (count / len(recovery_system.df)) * 100
        print(f"{strategy}: {count} borrowers ({percentage:.1f}%)")
else:
    print("Please run the complete analysis to see summary statistics.")